In [1]:
import numpy as np

from nanover.mdanalysis import frame_data_to_mdanalysis
from nanover.utilities.change_buffers import DictionaryChange

KNOT_TYING = ("../openmm/openmm_files/17-ala.xml", "knot-tying-checkpoints-better.nanover.zip")
KNOT_TYING_STRICT = ("../openmm/openmm_files/17-ala.xml", "knot-tying-strict-3.nanover.zip")
NANOTUBE = ("nanotube-methane.xml", "nanotube-keyframes.nanover.zip")
PATHS = KNOT_TYING

In [2]:
## Setup runner
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path(PATHS[0])
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="checkpoint replay")
imd_runner.load(0)

In [3]:
from nanover.mdanalysis import frame_data_to_mdanalysis

universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

In [4]:
from nanover.recording import NanoverRecordingReader
from tutorials.experiments.keyframes import extract_keyframes, TargetGroup, extract_keyframes_brute

with NanoverRecordingReader.from_path(PATHS[1]) as reader:
    KEYFRAMES = extract_keyframes(reader)
    # KEYFRAMES = extract_keyframes_brute(reader, universe)

In [5]:
simulation.use_pbc_wrapping = False

In [6]:
from nanover.websocket import NanoverImdClient

client = NanoverImdClient.from_runner(imd_runner)
with client.root_selection.modify() as selection:
    selection.renderer = "cartoon"

# with client.root_selection.modify() as selection:
#     selection.renderer = {
#         'scale': .2,
#         'render': 'liquorice',
#         'color': {
#             'type': 'cpk',
#             'scheme': 'nanover'
#         },
#     }

In [7]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

In [8]:
import random
import colorsys
from openmm import unit
from threading import RLock
from nanover.utilities.state_dictionary import DictionaryChange

from smearmd import SmearAgent, KeyFrame

KEYFRAME_INDEX = -1
AGENT: SmearAgent | None = None
SELECTIONS = []
LOCK = RLock()


def assign_atoms(keyframe: KeyFrame):
    frame = simulation.make_topology_frame()

    system = simulation.simulation.system
    masses = [
        system.getParticleMass(particle).value_in_unit(unit.dalton)
        for particle in range(system.getNumParticles())
    ]

    neighbours = { i: set() for i in range(frame.particle_count) }
    for a, b in frame.bond_pairs:
        neighbours[int(a)].add(int(b))
        neighbours[int(b)].add(int(a))

    measures = []

    for target in keyframe.targets:
        distances = { i: frame.particle_count for i in range(frame.particle_count) }
        for particle in target.particles:
            distances[particle] = 0

        queue = set(target.particles)
        while queue:
            particle = queue.pop()
            for neighbour in neighbours[particle]:
                if distances[neighbour] > distances[particle] + 1:
                    distances[neighbour] = distances[particle] + 1
                    queue.add(neighbour)

        measures.append(distances)

    groups = [set() for _ in measures]

    for particle in range(frame.particle_count):
        mind = min(distances[particle] for distances in measures)
        group = random.choice([i for i, distances in enumerate(measures) if distances[particle] == mind])
        groups[group].add(particle)

    for selection in client.selections:
        client.remove_selection(selection)

    masses = [sum(masses[particle] for particle in group) for group in groups]

    for i, group in enumerate(groups):
        with client.create_selection(f"{i}", list(group)).modify() as selection:
            # selection.renderer = {
            #     "color": colorsys.hsv_to_rgb(i/len(groups), .75, 1.0),
            # }
            selection.renderer = {
                "color": [masses[i] / max(masses), 1., 1.],
            }


def update_error(errors: list[tuple[TargetGroup, float]], targets) -> None:
    try:
        with utilities.objects as objects:
            for i, (target, error) in enumerate(errors):
                error = min(1.0, error)

                objects.update_shape(
                    i,
                    shape="sphere",
                    position=targets[i],
                    color=[1.0, 1.0-error, 1.0-error, 1.0],
                    size=0.1,
                )

                objects.update_shape(
                    i,
                    text=f"error: {error:.2g}",
                    position=targets[i],
                    color=[1.0, 1.0-error, 1.0-error, 1.0],
                    size=0.05,
                )

            objects.update_line(
                "backbone",
                text=f"error: {error:.2g}",
                positions=[targets[i] for i in range(len(errors))],
                color=[1.0, 1.0, 1.0, 1.0],
                size=0.05,
            )
    except Exception as e:
        with output:
            print(e)


def smear_next():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = KEYFRAME_INDEX + 1

    if KEYFRAME_INDEX == len(KEYFRAMES):
        smear_cancel()
        return

    utilities.notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_back():
    global KEYFRAME_INDEX
    KEYFRAME_INDEX = max(0, KEYFRAME_INDEX - 1)

    utilities.notify_all(f"MATCHING KEYFRAME {KEYFRAME_INDEX}")
    smear_match(KEYFRAMES[KEYFRAME_INDEX])


def smear_match(keyframe):
    global AGENT

    if AGENT is None:
        AGENT = SmearAgent.from_runner(imd_runner)
        AGENT.speed = .2 #.1 / 30
        AGENT.update.add_callback(update_error)

    # assign_atoms(keyframe)

    AGENT.set_keyframe(keyframe)
    AGENT.start()


def smear_cancel():
    global AGENT, KEYFRAME_INDEX

    if AGENT is None:
        return

    AGENT.close()
    AGENT = None
    KEYFRAME_INDEX = -1

    removals = set()
    removals |= {f"object.shape.{i}" for i in range (50)}
    removals |= {f"object.label.{i}" for i in range (50)}
    removals |= {"object.line.backbone"}
    imd_runner.app_server.update_state(DictionaryChange(removals=removals))

    utilities.notify_all(f"RESETTING AGENT")


imd_runner.app_server.register_command("user/smear/next", smear_next)
imd_runner.app_server.register_command("user/smear/back", smear_back)
imd_runner.app_server.register_command("user/smear/cancel", smear_cancel)

In [9]:
from ipywidgets import Output
output = Output()
output

Output()

In [10]:
with output:
    print("test")

In [11]:
from nanover.jupyter.nglclient import NGLClient

client = NGLClient.from_runner(imd_runner)
client.view

NGLWidget()

In [12]:
from nanover.jupyter.controls import show_runner_controls

show_runner_controls(imd_runner)

Button(description='Close Server', style=ButtonStyle())

interactive(children=(Dropdown(description='Force Type', index=1, options=(('Gaussian', 'gaussian'), ('Harmoni…

interactive(children=(IntSlider(value=100, description='Force Scale', max=1000, min=1), Output()), _dom_classe…

interactive(children=(FloatSlider(value=0.4, description='Force Range', max=2.0, min=0.1, step=0.05), Output()…

interactive(children=(FloatSlider(value=1.0, description='Passthrough', max=1.0), Output()), _dom_classes=('wi…